# pems06 按年节点增长数据处理

规则：
- 扫描 pems06 目录下 `d06_text_station_5min_YYYY_MM_DD.txt.gz`。
- **首年**：以数据集**首日**的 stations 作为该年的固定节点集合。
- **其余年份**：以每年 **1月1日** 当天的 stations 作为该年的节点集合。
- 每年只保留「该年锚定日」出现的节点，每年保存为一个 CSV。

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from collections import defaultdict

In [2]:
PEMS_DIR = './pems06'
OUTPUT_DIR = '.'
PREFIX = 'd06_text_station_5min'
SUFFIX = '.txt.gz'

COLUMN_NAMES = [
    "Timestamp", "Station", "District", "Freeway #",
    "Direction of Travel", "Lane Type", "Station Length",
    "Samples", "% Observed", "Total Flow", "Avg Occupancy", "Avg Speed"
]

## 1. 读取 pems06 目录，解析所有可用日期

In [3]:
def list_pems_dates(data_dir):
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"目录不存在: {data_dir}")
    dates = {}
    for f in os.listdir(data_dir):
        if not f.startswith(PREFIX) or not f.endswith(SUFFIX):
            continue
        try:
            rest = f[len(PREFIX):-len(SUFFIX)].strip(' _')
            parts = rest.split('_')
            if len(parts) == 3:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                dt = date(y, m, d)
                dates[dt] = os.path.join(data_dir, f)
        except (ValueError, IndexError):
            continue
    return dates

available = list_pems_dates(PEMS_DIR)
sorted_dates = sorted(available.keys())
first_date = min(sorted_dates)
print(f"pems06 共 {len(available)} 个日期文件")
print(f"首日: {first_date}, 末日: {sorted_dates[-1]}")

pems06 共 7425 个日期文件
首日: 2005-09-01, 末日: 2025-12-31


## 2. 确定每年的锚定日与节点集合

In [4]:
def get_anchor_date(year, first_date):
    """首年用数据集首日，其余年份用该年 1 月 1 日。"""
    if year == first_date.year:
        return first_date
    return date(year, 1, 1)

def read_station_ids_from_file(path):
    df = pd.read_csv(path, header=None, usecols=[1], names=['Station'], compression='gzip')
    return np.sort(df['Station'].dropna().unique().astype(int))

years = sorted(set(d.year for d in sorted_dates))
year_anchor = {}
year_nodes = {}

for y in years:
    anchor = get_anchor_date(y, first_date)
    year_anchor[y] = anchor
    if anchor not in available:
        print(f"警告: {y} 年锚定日 {anchor} 无数据文件，跳过该年")
        continue
    path = available[anchor]
    year_nodes[y] = read_station_ids_from_file(path)
    print(f"{y} 年 锚定日={anchor}, 节点数={len(year_nodes[y])}")

2005 年 锚定日=2005-09-01, 节点数=13
2006 年 锚定日=2006-01-01, 节点数=13
2007 年 锚定日=2007-01-01, 节点数=13
2008 年 锚定日=2008-01-01, 节点数=27
2009 年 锚定日=2009-01-01, 节点数=38
2010 年 锚定日=2010-01-01, 节点数=222
2011 年 锚定日=2011-01-01, 节点数=260
2012 年 锚定日=2012-01-01, 节点数=260
2013 年 锚定日=2013-01-01, 节点数=277
2014 年 锚定日=2014-01-01, 节点数=225
2015 年 锚定日=2015-01-01, 节点数=418
2016 年 锚定日=2016-01-01, 节点数=461
2017 年 锚定日=2017-01-01, 节点数=505
2018 年 锚定日=2018-01-01, 节点数=558
2019 年 锚定日=2019-01-01, 节点数=624
2020 年 锚定日=2020-01-01, 节点数=630
2021 年 锚定日=2021-01-01, 节点数=669
2022 年 锚定日=2022-01-01, 节点数=670
2023 年 锚定日=2023-01-01, 节点数=690
2024 年 锚定日=2024-01-01, 节点数=705
2025 年 锚定日=2025-01-01, 节点数=746


## 3. 按年汇总：只保留锚定日节点，每年保存一个 CSV

In [5]:
def read_one_day(path, columns=None):
    if columns is None:
        columns = list(range(12))
    df = pd.read_csv(path, header=None, usecols=columns,
                    names=[COLUMN_NAMES[i] for i in columns], compression='gzip')
    return df


def generate_empty_day_df(day, node_set):
    """当某天文件缺失时，为该天所有节点生成 5 分钟分辨率、全 0 的占位数据。"""
    # PeMS 5min: 24*60/5 = 288 个时间点
    start_dt = datetime(day.year, day.month, day.day, 0, 0)
    timestamps = [
        (start_dt + timedelta(minutes=5 * i)).strftime("%m/%d/%Y %H:%M:%S")
        for i in range(288)
    ]

    rows = []
    for ts in timestamps:
        for st in node_set:
            rows.append([
                ts,          # Timestamp
                st,          # Station
                np.nan,      # District
                np.nan,      # Freeway #
                np.nan,      # Direction of Travel
                np.nan,      # Lane Type
                np.nan,      # Station Length
                0,           # Samples
                0,           # % Observed
                0,           # Total Flow
                0,           # Avg Occupancy
                0,           # Avg Speed
            ])

    return pd.DataFrame(rows, columns=COLUMN_NAMES)


os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in years:
    if year not in year_nodes:
        continue
    node_set = set(year_nodes[year])
    start = first_date if year == first_date.year else date(year, 1, 1)
    end = date(year, 12, 31)
    chunks = []
    current = start
    while current <= end:
        if current in available:
            path = available[current]
            df = read_one_day(path)
            df = df[df['Station'].isin(node_set)]
        else:
            # 该天文件缺失，补 0
            df = generate_empty_day_df(current, node_set)
        chunks.append(df)
        current += timedelta(days=1)
    if not chunks:
        print(f"{year} 年无可用数据，跳过")
        continue
    out_df = pd.concat(chunks, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f'pems06_{year}.csv')
    out_df.to_csv(out_path, index=False)
    print(f"{year}: 节点数={len(node_set)}, 行数={len(out_df)}, 已保存 -> {out_path}")

2005: 节点数=13, 行数=456612, 已保存 -> .\pems06_2005.csv
2006: 节点数=13, 行数=1366248, 已保存 -> .\pems06_2006.csv
2007: 节点数=13, 行数=1366248, 已保存 -> .\pems06_2007.csv
2008: 节点数=27, 行数=2845368, 已保存 -> .\pems06_2008.csv
2009: 节点数=38, 行数=873084, 已保存 -> .\pems06_2009.csv
2010: 节点数=222, 行数=22974000, 已保存 -> .\pems06_2010.csv
2011: 节点数=260, 行数=27324960, 已保存 -> .\pems06_2011.csv
2012: 节点数=260, 行数=22821268, 已保存 -> .\pems06_2012.csv
2013: 节点数=277, 行数=27293146, 已保存 -> .\pems06_2013.csv
2014: 节点数=225, 行数=23054498, 已保存 -> .\pems06_2014.csv
2015: 节点数=418, 行数=43263537, 已保存 -> .\pems06_2015.csv
2016: 节点数=461, 行数=47418216, 已保存 -> .\pems06_2016.csv
2017: 节点数=505, 行数=52663336, 已保存 -> .\pems06_2017.csv
2018: 节点数=558, 行数=58146316, 已保存 -> .\pems06_2018.csv
2019: 节点数=624, 行数=65215924, 已保存 -> .\pems06_2019.csv
2020: 节点数=630, 行数=65205655, 已保存 -> .\pems06_2020.csv
2021: 节点数=669, 行数=69078044, 已保存 -> .\pems06_2021.csv
2022: 节点数=670, 行数=68609213, 已保存 -> .\pems06_2022.csv
2023: 节点数=690, 行数=71224901, 已保存 -> .\pems06_2023.csv
2024:

## 4. 各年节点数汇总

In [ ]:
summary = pd.DataFrame({
    'year': list(year_nodes.keys()),
    'anchor_date': [year_anchor[y] for y in year_nodes],
    'n_nodes': [len(year_nodes[y]) for y in year_nodes],
}).sort_values('year').reset_index(drop=True)
summary

,year,anchor_date,n_nodes
0,2005,2005-09-01,13
1,2006,2006-01-01,13
2,2007,2007-01-01,13
3,2008,2008-01-01,27
4,2009,2009-01-01,38
5,2010,2010-01-01,222
6,2011,2011-01-01,260
7,2012,2012-01-01,260
8,2013,2013-01-01,277
9,2014,2014-01-01,225


: 

## 5. Pivot 处理：行=5min时间步, 列=Station, 值=Total Flow

读取每年的 `pems06_{year}.csv`，pivot 成：
- **行**：完整的 5 分钟时间序列（缺失时间步自动补 NaN）
- **列**：每个 Station ID
- **值**：Total Flow

输出保存为 `pems06_{year}_flow.csv`

In [1]:
import os, re, glob
import pandas as pd
from datetime import date

FLOW_OUTPUT_DIR = '.'
FIRST_DATE = date(2005, 9, 1)
DISTRICT = 'pems06'

_csv_files = glob.glob(os.path.join(FLOW_OUTPUT_DIR, f'{DISTRICT}_[0-9][0-9][0-9][0-9].csv'))
years = sorted(int(re.search(rf'{DISTRICT}_(\d{{4}})\.csv', f).group(1)) for f in _csv_files)
print(f"扫描到 {len(years)} 个年度 CSV: {years[0]}~{years[-1]}")

def pivot_yearly_csv(year, first_date=FIRST_DATE, output_dir=FLOW_OUTPUT_DIR):
    src_path = os.path.join(output_dir, f'{DISTRICT}_{year}.csv')
    if not os.path.exists(src_path):
        print(f"  [跳过] {src_path} 不存在")
        return

    df = pd.read_csv(src_path, usecols=['Timestamp', 'Station', 'Total Flow'])
    print(f"  读取完成: {len(df):,} 行")

    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    pivot = df.pivot_table(
        index='Timestamp', columns='Station',
        values='Total Flow', aggfunc='sum'
    )
    del df

    pivot.columns = pivot.columns.astype(int)
    pivot = pivot[sorted(pivot.columns)]

    if year == first_date.year:
        start_dt = pd.Timestamp(first_date)
    else:
        start_dt = pd.Timestamp(year, 1, 1)
    end_dt = pd.Timestamp(year, 12, 31, 23, 55)

    full_idx = pd.date_range(start=start_dt, end=end_dt, freq='5min')
    pivot = pivot.reindex(full_idx)
    pivot.index.name = 'Timestamp'

    out_path = os.path.join(output_dir, f'{DISTRICT}_{year}_flow.csv')
    pivot.to_csv(out_path)

    n_nan_rows = pivot.isna().all(axis=1).sum()
    print(f"  {year}: stations={pivot.shape[1]}, "
          f"时间步={pivot.shape[0]}(期望{len(full_idx)}), "
          f"全NaN行={n_nan_rows}, 已保存 -> {out_path}")
    return pivot.shape

print("pivot_yearly_csv 函数已定义")

扫描到 21 个年度 CSV: 2005~2025
pivot_yearly_csv 函数已定义


In [2]:
import time

results = {}
for year in years:
    print(f"\n{'='*60}")
    print(f"处理 {year} 年 ...")
    t0 = time.time()
    shape = pivot_yearly_csv(year)
    elapsed = time.time() - t0
    print(f"  耗时: {elapsed:.1f}s")
    if shape:
        results[year] = shape

print(f"\n{'='*60}")
print(f"全部完成! 共处理 {len(results)} 年")


处理 2005 年 ...
  读取完成: 456,612 行
  2005: stations=13, 时间步=35136(期望35136), 全NaN行=12, 已保存 -> .\pems06_2005_flow.csv
  耗时: 2.1s

处理 2006 年 ...
  读取完成: 1,366,248 行
  2006: stations=13, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems06_2006_flow.csv
  耗时: 8.0s

处理 2007 年 ...
  读取完成: 1,366,248 行
  2007: stations=13, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems06_2007_flow.csv
  耗时: 9.3s

处理 2008 年 ...
  读取完成: 2,845,368 行
  2008: stations=27, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems06_2008_flow.csv
  耗时: 18.4s

处理 2009 年 ...
  读取完成: 873,084 行
  2009: stations=38, 时间步=105120(期望105120), 全NaN行=80940, 已保存 -> .\pems06_2009_flow.csv
  耗时: 5.0s

处理 2010 年 ...
  读取完成: 22,974,000 行
  2010: stations=222, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems06_2010_flow.csv
  耗时: 111.2s

处理 2011 年 ...
  读取完成: 27,324,960 行
  2011: stations=260, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems06_2011_flow.csv
  耗时: 196.3s

处理 2012 年 ...
  读取完成: 22,821,268 行
  2012: stations=260, 时间步=105408(期望105408), 全NaN行=13, 已

## 6. 检查输出结果汇总

In [3]:
summary_flow = pd.DataFrame([
    {'year': y, 'rows': s[0], 'stations': s[1]}
    for y, s in results.items()
])
summary_flow = summary_flow.sort_values('year').reset_index(drop=True)
print("各年 flow pivot 结果汇总:")
summary_flow

各年 flow pivot 结果汇总:


,year,rows,stations
0,2005,35136,13
1,2006,105120,13
2,2007,105120,13
3,2008,105408,27
4,2009,105120,38
5,2010,105120,222
6,2011,105120,260
7,2012,105408,260
8,2013,105120,277
9,2014,105120,225


In [4]:
check = pd.read_csv(f'./{DISTRICT}_{years[0]}_flow.csv', index_col=0, parse_dates=True, nrows=10)
print(f"{DISTRICT}_{years[0]}_flow.csv 前 10 行, shape={check.shape}")
check

pems06_2005_flow.csv 前 10 行, shape=(10, 13)


,600053,600054,600064,600065,600231,600232,600233,600276,600277,600279,600280,600281,600282
Timestamp,,,,,,,,,,,,,
2005-09-01 00:00:00,35.0,20.0,35.0,34.0,17.0,47.0,33.0,48.0,39.0,32.0,275.0,47.0,10.0
2005-09-01 00:05:00,39.0,41.0,55.0,44.0,24.0,45.0,27.0,39.0,34.0,29.0,282.0,37.0,27.0
2005-09-01 00:10:00,43.0,42.0,45.0,34.0,33.0,36.0,37.0,48.0,36.0,28.0,286.0,35.0,19.0
2005-09-01 00:15:00,40.0,45.0,32.0,29.0,21.0,51.0,41.0,37.0,32.0,19.0,263.0,36.0,29.0
2005-09-01 00:20:00,28.0,36.0,40.0,27.0,18.0,34.0,40.0,36.0,23.0,23.0,274.0,30.0,21.0
2005-09-01 00:25:00,27.0,26.0,37.0,25.0,20.0,28.0,28.0,40.0,33.0,15.0,258.0,37.0,16.0
2005-09-01 00:30:00,31.0,32.0,43.0,31.0,23.0,35.0,38.0,29.0,30.0,26.0,252.0,24.0,10.0
2005-09-01 00:35:00,28.0,29.0,45.0,26.0,21.0,38.0,34.0,40.0,30.0,21.0,274.0,34.0,16.0
2005-09-01 00:40:00,32.0,33.0,41.0,22.0,21.0,37.0,29.0,36.0,35.0,18.0,261.0,29.0,10.0
